# Figure 1 — Daily global RMSE

Global error evolution for four primary diagnostics.


## 1. Load only the required common-grid data


In [ ]:
from pathlib import Path
import importlib
import sys
import numpy as np
import xarray as xr
import dask
from dask.distributed import Client, get_client

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError('Run from ml_implement_paper/ or its notebooks/ directory')
sys.path.insert(0, str(PROJECT_ROOT))

from src import io as data_io
from src import metrics, plotting, preprocessing
plotting = importlib.reload(plotting)

config = data_io.load_config(PROJECT_ROOT / 'config' / 'paths.yaml')
analysis = config['analysis']

# ------------------------- User-adjustable settings -------------------------
OUTPUT_ROOT = Path('/global/cfs/cdirs/e3sm/www/zhan391/sea_crogs/online_diag')
ML_ROOT = Path('/pscratch/sd/z/zhan391/seacrogs_scratch/ml_method_2026')
REFERENCE_ROOT = Path('/pscratch/sd/z/zhan391/seacrogs_scratch/reference_nudge')
POST_SUBDIR = Path('post/atm/180x360_aave/ts/3hourly/1yr')
PERIOD = '201201_201212'
FORCE_COMPUTE = False  # True: overwrite selected case caches.
REQUIRED_VARIABLES = ['T500', 'U200', 'U850', 'TMQ', 'PRECT']
paths = {
    'processed': OUTPUT_ROOT / 'processed',
    'figures': OUTPUT_ROOT / 'figures',
    'tables': OUTPUT_ROOT / 'tables',
}
for output_dir in paths.values():
    output_dir.mkdir(parents=True, exist_ok=True)

# Select cases here: keep CTRL and REF, and comment out any ML case you do not want.
CASE_DIRS = {
    'CTRL': ML_ROOT / 'F20TR_NE30PG2_EC30TO60E2R2_CTRL',
    'UNET-IMT': ML_ROOT / 'F20TR_NE30PG2_EC30TO60E2R2_UVTQ_UNET_IMT_NS6_WOQMADJ_WOTVCON_PBL222_WOVSMOOTH',
    'UNETXTR-IMT': ML_ROOT / 'F20TR_NE30PG2_EC30TO60E2R2_UVTQ_UNETXTR_IMT_NS6_WOQMADJ_WOTVCON_PBL222_WOVSMOOTH',
    'UNETXTR-IMT-A15': ML_ROOT / 'F20TR_NE30PG2_EC30TO60E2R2_UVTQ_UNETXTR_IMT_NS6_WQMADJ_WTVCON_PBL222_WVSMOOTH-S0.15',
    'UNETXTR-IMT-C05': ML_ROOT / 'F20TR_NE30PG2_EC30TO60E2R2_UVTQ_UNETXTR_IMT_NS6_SCL0.5_WOQMADJ_WOTVCON_PBL222_WOVSMOOTH',
    'UNETXTR-IMT-C05A15': ML_ROOT / 'F20TR_NE30PG2_EC30TO60E2R2_UVTQ_UNETXTR_IMT_NS6_SCL0.5_WQMADJ_WTVCON_PBL222_WVSMOOTH-S0.15',
    'UNETXTR-LCZ-C05A15': ML_ROOT / 'F20TR_NE30PG2_EC30TO60E2R2_UVTQ_UNETXTR_LANCZOS_NS6_SCL0.5_WQMADJ_WTVCON_PBL222_WVSMOOTH-S0.15',
    'UNETXTR-LCZ-C05A15-PTAP100': ML_ROOT / 'F20TR_NE30PG2_EC30TO60E2R2_UVTQ_UNETXTR_LANCZOS_NS6_SCL0.5_PTAPUNI100HPA_WQMADJ_WTVCON_PBL222_WVSMOOTH-S0.15',
    'REF': REFERENCE_ROOT / 'F20TR_ne30pg2_EC30to60E2r2_NDGUVTQ_IMT_3hr_pm-cpu_08-01-25',
}
FONT_SIZE = 14
FIGURE_SIZE = (12, 10)
FIGURE_LAYOUT = (3, 2)
LINE_WIDTH = 1.5
LEGEND_LOCATION = 'best'
LEGEND_FRAME = False
FIGURE_TITLE = 'Daily global atmospheric RMSE'
X_AXIS_LABEL = 'Date'
SHARE_X_AXIS = True
COLORS = {
    'CTRL': '#222222', 'UNET-IMT': '#2878B5', 'UNETXTR-IMT': '#D95319',
    'UNETXTR-IMT-A15': '#9467BD', 'UNETXTR-IMT-C05': '#2CA02C',
    'UNETXTR-IMT-C05A15': '#8C564B', 'UNETXTR-LCZ-C05A15': '#17BECF',
    'UNETXTR-LCZ-C05A15-PTAP100': '#E377C2',
}
# ---------------------------------------------------------------------------

DIAGNOSTIC_DIR = paths['processed'] / 'global_rmse_timeseries'
DIAGNOSTIC_DIR.mkdir(parents=True, exist_ok=True)
ML_CASES = tuple(name for name in CASE_DIRS if name not in {'CTRL', 'REF'})
ANALYSIS_CASES = ('CTRL', *ML_CASES)
DIAGNOSTIC_PATHS = {
    name: DIAGNOSTIC_DIR / f'{name}_{PERIOD}.nc' for name in ANALYSIS_CASES
}

def cache_is_compatible(path, experiment):
    if not path.exists():
        return False
    try:
        with xr.open_dataset(path) as cached:
            variables = set(cached['variable'].values.astype(str))
            experiments = set(cached['experiment'].values.astype(str))
            return (set(REQUIRED_VARIABLES).issubset(variables)
                    and experiment in experiments
                    and {'rmse', 'improvement_percent'}.issubset(cached.data_vars))
    except (KeyError, OSError, ValueError):
        return False

CASES_TO_COMPUTE = tuple(
    name for name, path in DIAGNOSTIC_PATHS.items()
    if FORCE_COMPUTE or not cache_is_compatible(path, name)
)
RECOMPUTE = bool(CASES_TO_COMPUTE)

if RECOMPUTE:
    # Reuse an existing client or start a conservative local threaded client.
    try:
        client = get_client()
    except ValueError:
        client = Client(n_workers=4, threads_per_worker=1, processes=False, dashboard_address=':0')

    INPUT_CASES = ('REF', *CASES_TO_COMPUTE)
    files = {
        name: [case_dir / POST_SUBDIR / f'{variable}_{PERIOD}.nc' for variable in REQUIRED_VARIABLES]
        for name, case_dir in CASE_DIRS.items() if name in INPUT_CASES
    }
    missing = {name: [str(path) for path in paths_ if not path.exists()] for name, paths_ in files.items()}
    missing = {name: paths_ for name, paths_ in missing.items() if paths_}
    if missing:
        details = '\n'.join(f'  {name}: {len(paths_)} missing file(s)' for name, paths_ in missing.items())
        raise FileNotFoundError('Postprocess the required variables first:\n' + details)

    chunks = {'time': 32, 'lat': 45, 'lon': 90}
    datasets = {
        name: xr.merge(
            [xr.open_dataset(path, chunks=chunks, cache=False) for path in paths_],
            join='exact',
            compat='no_conflicts',
        )
        for name, paths_ in files.items()
    }
    datasets = {
        name: preprocessing.subset_time(ds, analysis['start_date'], analysis['end_date'])
        for name, ds in datasets.items()
    }
    datasets = preprocessing.match_common_times(datasets)
    datasets = {name: preprocessing.daily_mean(ds) for name, ds in datasets.items()}

    sample = datasets[next(iter(datasets))][REQUIRED_VARIABLES[0]].isel(time=0, drop=True)
    area = np.cos(np.deg2rad(sample['lat'])).clip(min=0).broadcast_like(sample)
    area = area / area.sum()
    status = {
        'mode': 'compute', 'cases': CASES_TO_COMPUTE,
        'datasets': {name: dict(ds.sizes) for name, ds in datasets.items()},
    }
else:
    status = {'mode': 'cached', 'paths': {name: str(path) for name, path in DIAGNOSTIC_PATHS.items()}}
status

## 2. Quality control


In [2]:
# Run all finite-value checks together so Dask can share I/O efficiently.

qc = {}
if RECOMPUTE:
    qc_keys = []
    qc_tasks = []
    for case_name, dataset in datasets.items():
        for variable in REQUIRED_VARIABLES:
            qc_keys.append((case_name, variable))
            qc_tasks.extend([
                np.isfinite(dataset[variable]).any().data,
                (~np.isfinite(dataset[variable])).sum().data,
            ])
    qc_values = dask.compute(*qc_tasks)
    for index, key in enumerate(qc_keys):
        has_finite = bool(qc_values[2 * index])
        invalid_count = int(qc_values[2 * index + 1])
        if not has_finite:
            raise ValueError(f'{key[0]}:{key[1]} contains no finite values')
        qc.setdefault(key[0], {})[key[1]] = invalid_count
else:
    qc = {'status': 'skipped; using cached diagnostic'}
qc

/global/homes/z/zhan391/.conda/envs/e3sm_analysis/lib/python3.14/site-packages/distributed/client.py:3398: UserWarning: Sending large graph of size 91.52 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


{'CTRL': {'T500': 0, 'U200': 0, 'U850': 0, 'TMQ': 0, 'PRECT': 0},
 'UNET-IMT': {'T500': 0, 'U200': 0, 'U850': 0, 'TMQ': 0, 'PRECT': 0},
 'UNETXTR-IMT': {'T500': 0, 'U200': 0, 'U850': 0, 'TMQ': 0, 'PRECT': 0},
 'UNETXTR-IMT-A15': {'T500': 0, 'U200': 0, 'U850': 0, 'TMQ': 0, 'PRECT': 0},
 'UNETXTR-IMT-C05': {'T500': 0, 'U200': 0, 'U850': 0, 'TMQ': 0, 'PRECT': 0},
 'UNETXTR-IMT-C05A15': {'T500': 0, 'U200': 0, 'U850': 0, 'TMQ': 0, 'PRECT': 0},
 'UNETXTR-LCZ-C05A15': {'T500': 0, 'U200': 0, 'U850': 0, 'TMQ': 0, 'PRECT': 0},
 'REF': {'T500': 0, 'U200': 0, 'U850': 0, 'TMQ': 0, 'PRECT': 0}}

## 3. Process and save the diagnostic data


In [ ]:
if RECOMPUTE:
    reference = datasets['REF']
    control_rmse = None
    if 'CTRL' not in CASES_TO_COMPUTE:
        with xr.open_dataset(DIAGNOSTIC_PATHS['CTRL']) as cached_control:
            control_rmse = cached_control['rmse'].sel(experiment='CTRL', drop=True).load()
    for name in CASES_TO_COMPUTE:
        values = [
            metrics.weighted_rmse(
                datasets[name][variable], reference[variable], area, list(area.dims)
            )
            for variable in REQUIRED_VARIABLES
        ]
        case_rmse = xr.concat(
            values, dim=xr.IndexVariable('variable', REQUIRED_VARIABLES)
        ).rename('rmse')
        if name == 'CTRL':
            control_rmse = case_rmse
            improvement = xr.full_like(case_rmse, np.nan).rename('improvement_percent')
        else:
            improvement = metrics.percentage_improvement(
                case_rmse, control_rmse, analysis['minimum_control_rmse']
            ).rename('improvement_percent')
        case_diagnostic = xr.Dataset({
            'rmse': case_rmse, 'improvement_percent': improvement,
        }).expand_dims(experiment=[name])
        case_diagnostic.attrs.update({
            'experiment': name, 'period': PERIOD,
            'reference_case': str(CASE_DIRS['REF']),
        })
        data_io.save_dataset(case_diagnostic, DIAGNOSTIC_PATHS[name])

diagnostic = xr.concat(
    [xr.open_dataset(DIAGNOSTIC_PATHS[name]) for name in ANALYSIS_CASES], dim='experiment'
)
diagnostic

/global/homes/z/zhan391/.conda/envs/e3sm_analysis/lib/python3.14/site-packages/distributed/client.py:3398: UserWarning: Sending large graph of size 203.37 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


## 4. Reload the diagnostic product and create the figure


In [ ]:
FIGURE_PATH = paths['figures'] / 'fig01_rmse_timeseries.png'
PLOT_VARIABLES = REQUIRED_VARIABLES
PLOT_OPTIONS = {
    'figsize': FIGURE_SIZE,
    'layout': FIGURE_LAYOUT,
    'sharex': SHARE_X_AXIS,
    'colors': COLORS,
    'title': FIGURE_TITLE,
    'xlabel': X_AXIS_LABEL,
    'legend_loc': LEGEND_LOCATION,
    'legend_frame': LEGEND_FRAME,
    'linewidth': LINE_WIDTH,
    'font_size': FONT_SIZE,
}

diagnostic = xr.concat(
    [xr.open_dataset(DIAGNOSTIC_PATHS[name]) for name in ANALYSIS_CASES], dim='experiment'
)
plotting.plot_rmse_timeseries(diagnostic, PLOT_VARIABLES, FIGURE_PATH, **PLOT_OPTIONS)